In [1]:
from __future__ import annotations
# Sem essa linha, você não pode usar uma classe como tipo dentro dela mesma, porque a classe ainda não terminou de ser definida quando o interpretador lê o método.
from dataclasses import dataclass
# from dataclasses import dataclass
from typing import Callable, Iterable, List, Optional, Tuple, Union
# usar esses nomes ajuda o VS Code (ou o PyCharm) a te dar sugestões melhores e evita que você cometa erros bobos, como passar um número onde deveria ser uma lista.
import numpy as np

#Type Aliases (apelidos de tipo).
Number = Union[int, float]
ArrayLike = Union[Number, np.ndarray]

# Quando você define uma dataclass como frozen=True, você está criando um objeto imutável.
@dataclass(frozen=True)
class Impedancia:
    """
    Impedance object defined by a function Z(f_hz) -> complex.
    You can combine impedances in series/parallel and scan resonance points.
    """
    z_func: Callable[[np.ndarray], np.ndarray]
    label: str = ""

    # ----------------------------
    # Core evaluation
    # ----------------------------
    def __call__(self, f_hz: ArrayLike) -> np.ndarray:
        f = np.asarray(f_hz, dtype=float)

        if np.any(~np.isfinite(f)):
            raise ValueError("f_hz must be finite.")
        if np.any(f <= 0.0):
            raise ValueError("f_hz must be > 0.")

        z = self.z_func(f)
        z = np.asarray(z, dtype=complex)

        return z

    def at(self, f_hz: Number) -> complex:
        z = self(f_hz)
        return complex(np.atleast_1d(z)[0])

    # ----------------------------
    # Convenience getters
    # ----------------------------
    def real(self, f_hz: ArrayLike) -> np.ndarray:
        return np.real(self(f_hz))

    def imag(self, f_hz: ArrayLike) -> np.ndarray:
        return np.imag(self(f_hz))

    def mag(self, f_hz: ArrayLike) -> np.ndarray:
        return np.abs(self(f_hz))

    def phase_rad(self, f_hz: ArrayLike) -> np.ndarray:
        return np.angle(self(f_hz))

    def phase_deg(self, f_hz: ArrayLike) -> np.ndarray:
        return np.degrees(self.phase_rad(f_hz))

    def admittance(self) -> "Impedancia":
        def y_func(f: np.ndarray) -> np.ndarray:
            z = self.z_func(f)
            z = np.asarray(z, dtype=complex)
            if np.any(z == 0.0):
                raise ZeroDivisionError("Z(f) is zero at some frequency; cannot invert.")
            y = 1.0 / z
            return y

        new_label = f"1/({self.label})" if self.label else "1/Z"
        return Impedancia(z_func=y_func, label=new_label)

    # ----------------------------
    # Primitive elements
    # ----------------------------
    @staticmethod
    def resistor(R_ohm: Number, label: str = "") -> "Impedancia":
        if R_ohm < 0.0:
            raise ValueError("R_ohm must be >= 0.")

        def z_func(f: np.ndarray) -> np.ndarray:
            _ = f  # Frequency independent
            return (R_ohm + 0.0j) * np.ones_like(f, dtype=complex)

        base_label = label if label else f"R={R_ohm}Ω"
        return Impedancia(z_func=z_func, label=base_label)

    @staticmethod
    def inductor(L_h: Number, label: str = "") -> "Impedancia":
        if L_h < 0.0:
            raise ValueError("L_h must be >= 0.")

        def z_func(f: np.ndarray) -> np.ndarray:
            w = 2.0 * np.pi * f
            return 1.0j * w * L_h

        base_label = label if label else f"L={L_h}H"
        return Impedancia(z_func=z_func, label=base_label)

    @staticmethod
    def capacitor(C_f: Number, label: str = "") -> "Impedancia":
        if C_f <= 0.0:
            raise ValueError("C_f must be > 0.")

        def z_func(f: np.ndarray) -> np.ndarray:
            w = 2.0 * np.pi * f
            return 1.0 / (1.0j * w * C_f)

        base_label = label if label else f"C={C_f}F"
        return Impedancia(z_func=z_func, label=base_label)

    # ----------------------------
    # Combinations
    # ----------------------------
    @staticmethod
    def series(*items: "Impedancia", label: str = "") -> "Impedancia":
        if len(items) == 0:
            raise ValueError("Series requires at least one impedance.")

        def z_func(f: np.ndarray) -> np.ndarray:
            z_total = np.zeros_like(f, dtype=complex)
            for it in items:
                z_total = z_total + it(f)
            return z_total

        if label:
            base_label = label
        else:
            parts = [it.label for it in items if it.label]
            base_label = " + ".join(parts) if len(parts) > 0 else "Z_series"

        return Impedancia(z_func=z_func, label=base_label)

    @staticmethod
    def parallel(*items: "Impedancia", label: str = "", eps: float = 1e-30) -> "Impedancia":
        if len(items) == 0:
            raise ValueError("Parallel requires at least one impedance.")

        def z_func(f: np.ndarray) -> np.ndarray:
            y_total = np.zeros_like(f, dtype=complex)

            for it in items:
                z = it(f)
                if np.any(np.abs(z) <= eps):
                    raise ZeroDivisionError("One branch has |Z| ~ 0 at some frequency; parallel undefined.")
                y_total = y_total + (1.0 / z)

            if np.any(np.abs(y_total) <= eps):
                raise ZeroDivisionError("Total |Y| ~ 0 at some frequency; cannot invert.")

            return 1.0 / y_total

        if label:
            base_label = label
        else:
            parts = [it.label for it in items if it.label]
            if len(parts) > 0:
                base_label = " || ".join(parts)
            else:
                base_label = "Z_parallel"

        return Impedancia(z_func=z_func, label=base_label)


    # ----------------------------
    # Resonance helpers
    # ----------------------------
    @staticmethod
    def rlc_series(R_ohm: Number, L_h: Number, C_f: Number, label: str = "") -> "Impedancia":
        z_r = Impedancia.resistor(R_ohm)
        z_l = Impedancia.inductor(L_h)
        z_c = Impedancia.capacitor(C_f)

        base_label = label if label else f"Series RLC (R={R_ohm}, L={L_h}, C={C_f})"
        return Impedancia.series(z_r, z_l, z_c, label=base_label)

    @staticmethod
    def rlc_parallel(R_ohm: Number, L_h: Number, C_f: Number, label: str = "") -> "Impedancia":
        z_r = Impedancia.resistor(R_ohm)
        z_l = Impedancia.inductor(L_h)
        z_c = Impedancia.capacitor(C_f)

        base_label = label if label else f"Parallel RLC (R={R_ohm}, L={L_h}, C={C_f})"
        return Impedancia.parallel(z_r, z_l, z_c, label=base_label)

    @staticmethod
    def f0_lc(L_h: Number, C_f: Number) -> float:
        if L_h <= 0.0:
            raise ValueError("L_h must be > 0.")
        if C_f <= 0.0:
            raise ValueError("C_f must be > 0.")
        w0 = 1.0 / np.sqrt(L_h * C_f)
        f0 = w0 / (2.0 * np.pi)
        return float(f0)

    def find_resonances(
        self,
        f_min_hz: Number,
        f_max_hz: Number,
        n_points: int = 5000,
        mode: str = "min_absZ",
        top_k: int = 3,
        min_separation_hz: Optional[Number] = None,
    ) -> List[Tuple[float, float]]:
        """
        Returns a list of (f_hz, metric_value).
        mode:
          - "min_absZ": local minima of |Z|
          - "max_absZ": local maxima of |Z|
          - "min_absY": local minima of |Y| = |1/Z|
          - "max_absY": local maxima of |Y|
        """
        if f_min_hz <= 0.0:
            raise ValueError("f_min_hz must be > 0.")
        if f_max_hz <= f_min_hz:
            raise ValueError("f_max_hz must be > f_min_hz.")
        if n_points < 100:
            raise ValueError("n_points must be >= 100.")
        if top_k < 1:
            raise ValueError("top_k must be >= 1.")

        f = np.logspace(np.log10(float(f_min_hz)), np.log10(float(f_max_hz)), int(n_points))

        if mode == "min_absZ":
            metric = np.abs(self(f))
            want = "min"
        elif mode == "max_absZ":
            metric = np.abs(self(f))
            want = "max"
        elif mode == "min_absY":
            metric = np.abs(self.admittance()(f))
            want = "min"
        elif mode == "max_absY":
            metric = np.abs(self.admittance()(f))
            want = "max"
        else:
            raise ValueError('mode must be one of: "min_absZ", "max_absZ", "min_absY", "max_absY".')

        dm = np.diff(metric)

        if want == "min":
            idx = np.where((dm[:-1] < 0.0) & (dm[1:] > 0.0))[0] + 1
        else:
            idx = np.where((dm[:-1] > 0.0) & (dm[1:] < 0.0))[0] + 1

        candidates: List[Tuple[float, float]] = []
        for i in idx:
            candidates.append((float(f[i]), float(metric[i])))

        if len(candidates) == 0:
            best_i = int(np.argmin(metric)) if want == "min" else int(np.argmax(metric))
            return [(float(f[best_i]), float(metric[best_i]))]

        candidates.sort(key=lambda x: x[1], reverse=(want == "max"))
        selected: List[Tuple[float, float]] = []

        if min_separation_hz is None:
            min_sep = 0.0
        else:
            min_sep = float(min_separation_hz)

        for cand in candidates:
            if len(selected) >= top_k:
                break

            ok = True
            for sel in selected:
                if abs(cand[0] - sel[0]) < min_sep:
                    ok = False
            if ok:
                selected.append(cand)

        return selected


# Optional ASCII alias (useful on older toolchains / editors)
Impedancia = Impedancia


In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Parameters filtro sintonizado
R_filter = 1.27e-3
L_filter = 34.3e-3
C_filter = 8.54e-6

# Parameters externo na base do filtro
R_trafo = [23.8e-3, 23.8e-3, 23.8e-3]
L_trafo = [3.21e-3, 3.21e-3, 3.21e-3]

C_linha = 50e-6

# Filtro interno (RLC serie)
Z_filter = Impedancia.series(
    Impedancia.resistor(R_filter),
    Impedancia.inductor(L_filter),
    Impedancia.capacitor(C_filter),
)

# Lista de impedancias RL externas
Z_trafos = []
for i in range(len(R_trafo)):
    Z_tmp = Impedancia.series(
        Impedancia.resistor(R_trafo[i]),
        Impedancia.inductor(L_trafo[i]),
    )
    Z_trafos.append(Z_tmp)

# Linha
Z_linha = Impedancia.capacitor(C_linha)

# Paralelo: linha || Z_1 || Z_2
Z_eq_externo = Impedancia.parallel(Z_linha, Z_trafos[1], Z_trafos[2])

# Serie: Z_eq_externo + Z_0 + Z_filter
Z_eq = Impedancia.series(Z_eq_externo, Z_trafos[0], Z_filter)

# ----------------------------
# Frequency sweep
# ----------------------------
f_min = 60
f_max = 3000
f = np.logspace(np.log10(f_min), np.log10(f_max), 2000)

# ----------------------------
# Evaluate all impedances
# ----------------------------
Z_dict = {}
Z_dict["Z_filter"] = Z_filter(f)
Z_dict["Z_linha"] = Z_linha(f)
Z_dict["Z_trafo_0"] = Z_trafos[0](f)
Z_dict["Z_trafo_1"] = Z_trafos[1](f)
Z_dict["Z_trafo_2"] = Z_trafos[2](f)
Z_dict["Z_eq_externo"] = Z_eq_externo(f)
Z_dict["Z_eq"] = Z_eq(f)

# ----------------------------
# Figure 1: Magnitude + Phase (same colors)
# ----------------------------
fig_mp = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Magnitude |Z|", "Phase (deg)"),
)

alpha = 0.70

color_list = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown
    "#e377c2",  # pink
    "#7f7f7f",  # gray
    "#bcbd22",  # olive
    "#17becf",  # cyan
]

names = list(Z_dict.keys())

for i in range(len(names)):
    name = names[i]
    Zf = Z_dict[name]

    mag = np.abs(Zf)
    phase = np.angle(Zf, deg=True)

    color = color_list[i % len(color_list)]
    legend_group = name

    # Magnitude trace (shows legend)
    fig_mp.add_trace(
        go.Scatter(
            x=f,
            y=mag,
            mode="lines",
            name=name,
            opacity=alpha,
            line={"color": color},
            legendgroup=legend_group,
            showlegend=True,
        ),
        row=1,
        col=1,
    )

    # Phase trace (same color, same legendgroup, no duplicate legend)
    fig_mp.add_trace(
        go.Scatter(
            x=f,
            y=phase,
            mode="lines",
            name=name,
            opacity=alpha,
            line={"color": color},
            legendgroup=legend_group,
            showlegend=False,
        ),
        row=2,
        col=1,
    )

fig_mp.update_xaxes(type="log")
fig_mp.update_xaxes(title_text="Frequency (Hz)", row=2, col=1)

fig_mp.update_yaxes(type="log", title_text="|Z| (ohm)", row=1, col=1)
fig_mp.update_yaxes(title_text="Phase (deg)", row=2, col=1)

fig_mp.update_layout(
    height=850,
    hovermode="x unified",
    legend_title_text="Impedances",
)

fig_mp.show()

# ----------------------------
# Figure 2: Imaginary part only
# ----------------------------
fig_im = go.Figure()

for i in range(len(names)):
    name = names[i]
    Zf = Z_dict[name]

    imag_part = np.imag(Zf)

    color = color_list[i % len(color_list)]

    fig_im.add_trace(
        go.Scatter(
            x=f,
            y=imag_part,
            mode="lines",
            name=name,
            opacity=alpha,
            line={"color": color},
        )
    )

fig_im.update_xaxes(type="log", title_text="Frequency (Hz)")
fig_im.update_yaxes(title_text="Imag(Z) (ohm)")
fig_im.update_layout(
    height=500,
    hovermode="x unified",
    legend_title_text="Impedances",
)

fig_im.show()
